# RNA-Seq Post-Counted Processing Workflow
### Authors: Emily Skates, Stephen Williams

This script takes the output of STAR + featureCounts and performs a series of downstream analyses:
- **Phase 1:** Data Loading, DESeq2 Setup, and Dispersion Diagnostics
- **Phase 2:** Sample Clustering (PCA) & Group Differences (PERMANOVA)
- **Phase 3:** Differential Expression Analysis (DESeq2), Volcano, and MA Plots
- **Phase 4:** Co-expression Network Analysis (WGCNA)
- **Phase 5:** Gene Ontology (GO) Enrichment Analysis

## Environment Setup & Libraries
Assumes packages are installed. If not, use `install.packages("NAME")` or `BiocManager::install("NAME")`.

NOTE: The working directory needs to be set here to match where the files are stored

In [ ]:
setwd("/Users/steve/Library/CloudStorage/GoogleDrive-stev3.w1l@gmail.com/My Drive/Academia/c_collaborations/emilyS/")

library(DESeq2)
library(ggplot2)
library(ggvenn)
library(WGCNA)
library(pheatmap)
library(vegan)
library(pairwiseAdonis)
library(glue)
library(scales)
library(gtools)
library(clusterProfiler)
library(enrichplot)
library(org.Mm.eg.db)

## Global Configuration & Thresholds
Define your comparisons, gene filtering criteria, and plotting cut-offs here.

In [ ]:
copy_threshold <- 10        # Number of gene copies required to be considered expressed
condition_threshold <- 1    # Number of samples in a condition that must meet the copy threshold
PCA_gene_count <- 500       # Top variable genes for PCA
network_size <- 100         # Top variable genes for co-expression network
FC_thresh <- 1.5            # Log2 fold change threshold for DE
FCPadj <- 0.05              # Adjusted p-value threshold for DE (Tier 1)
FCPval <- 0.05              # Non-adjusted p-value threshold for DE (Tier 2)

## Phase 1a: Load Data & perform DESeq2 Setup

In [ ]:
file_list <- Sys.glob("o_outputs/sample_*/gene_counts.txt")
file_list <- mixedsort(file_list)

temp_first <- read.table(file_list[1], header = TRUE, skip = 1, stringsAsFactors = FALSE)
count_matrix <- data.frame(Geneid = temp_first[, 1])

for (file in file_list) {
    sample_name <- basename(dirname(file))
    temp_data <- read.table(file, header = TRUE, skip = 1, stringsAsFactors = FALSE)
    count_matrix[[sample_name]] <- temp_data[, 7]
}

row.names(count_matrix) <- count_matrix$Geneid
count_matrix$Geneid <- NULL

sample_info <- data.frame(
    Condition1 = factor(c(rep("unlab", 3), rep("lab", 3), rep("unlab", 3), rep("lab", 3)), 
                        levels = c("unlab", "lab")),
    Condition2 = factor(c(rep("pull_down", 6), rep("input", 6)), 
                        levels = c("input", "pull_down"))
)

sample_info$SubjectNested <- factor(rep(c("Subj1", "Subj2", "Subj3", "Subj1", "Subj2", "Subj3"), times = 2)) # XXX

row.names(sample_info) <- colnames(count_matrix)

# The standard formula for a nested paired design:
# We control for the nested subject effect within each treatment group, 
# plus the main effect of Fraction (Condition2), plus the Interaction.
# In DESeq2, this is structured cleanly by adding a nested term:
dds <- DESeqDataSetFromMatrix(countData = count_matrix,
                              colData = sample_info,
                              design = ~ Condition1 + Condition1:SubjectNested + Condition2 + Condition1:Condition2)

keep <- rowSums(counts(dds) >= copy_threshold) >= condition_threshold
dds <- dds[keep, ]
vsd <- vst(dds, blind = FALSE) # Variance Stabilizing Transformation

## Phase 1b: Dispersion Diagnostics
Generates diagnostic plots to assess the data fit.

In [ ]:
dds <- dds |>
    estimateSizeFactors() |>
    estimateDispersions()

disp_df <- data.frame(
    baseMean = mcols(dds)$baseMean,
    dispGeneEst = mcols(dds)$dispGeneEst,
    dispersion = mcols(dds)$dispersion,
    dispFit = mcols(dds)$dispFit
)
disp_df <- disp_df[disp_df$baseMean > 0, ]

dispersion_plot <- ggplot(disp_df, aes(x = baseMean)) +
    geom_point(aes(y = dispGeneEst), color = "black", alpha = 0.3, size = 1) +
    geom_point(aes(y = dispersion), color = "dodgerblue", alpha = 0.3, size = 1, shape = 1) +
    geom_line(aes(y = dispFit), color = "red", linewidth = 1) +
    scale_x_log10() +
    scale_y_log10() +
    theme_minimal() +
    labs(title = "DESeq2 Dispersion Estimates Diagnostics",
         x = "Mean of Normalized Counts",
         y = "Dispersion")

print(dispersion_plot)
ggsave(filename = glue("f_figures/Dispersion_Estimates.png"),
       plot = dispersion_plot, width = 8, height = 6, dpi = 300)

## Phase 2a: PCA (Checking Sample Clustering)

In [ ]:
# Change "Group" to c("Condition1", "Condition2")
pca_data <- plotPCA(vsd, intgroup = c("Condition1", "Condition2"), ntop = PCA_gene_count, returnData = TRUE)

pca_figure <- ggplot(pca_data, aes(x = PC1, y = PC2, color = group, shape = group)) +
    geom_point(size = 2.5) +
    theme_minimal() +
    scale_color_manual(
        name = "Experimental Condition",
        values = c(
            "lab:input"        = "#D55E00", 
            "lab:pull_down"    = "#009E73", 
            "unlab:input"      = "#56B4E9", 
            "unlab:pull_down"  = "#CC79A7"  
        ),
        labels = c(
            "lab:input"        = "Labeled Input",
            "lab:pull_down"    = "Labeled Pull-Down",
            "unlab:input"      = "Unlabeled Input",
            "unlab:pull_down"  = "Unlabeled Pull-Down"
        )
    ) +
    scale_shape_manual(
        name = "Experimental Condition",
        values = c(
            "lab:input"        = 16,
            "lab:pull_down"    = 17,
            "unlab:input"      = 15,
            "unlab:pull_down"  = 18  
        ),
        labels = c(
            "lab:input"        = "Labeled Input",
            "lab:pull_down"    = "Labeled Pull-Down",
            "unlab:input"      = "Unlabeled Input",
            "unlab:pull_down"  = "Unlabeled Pull-Down"
        )
    ) +
    labs(
        x = paste0("PC1: ", round(attr(pca_data, "percentVar")[1] * 100, 1), "% variance"),
        y = paste0("PC2: ", round(attr(pca_data, "percentVar")[2] * 100, 1), "% variance")
    ) +
    theme(
        axis.text = element_text(size = 10),
        axis.title = element_text(size = 14),
        panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5)
    )

print(pca_figure)

## Phase 2b: PERMANOVA + Beta Diversity

In [ ]:
mat_vst <- t(assay(vsd))
dist_matrix <- vegdist(mat_vst, method = "euclidean")

# --- 1. Full Multi-Factor PERMANOVA ---
# This evaluates the main effects of Labeled vs Unlabeled (Condition1),
# Input vs Pulldown (Condition2), and importantly, their interaction (*)
permanova_results1 <- adonis2(dist_matrix ~ Condition1 * Condition2, data = sample_info, permutations = 999)
print("--- Multi-Factor PERMANOVA (Condition1 * Condition2) ---")
print(permanova_results1)
print(" ")

# --- 2. Main Effect of Fraction Only ---
# Keeps your original look at just Pulldown vs Input across all samples
permanova_results2 <- adonis2(dist_matrix ~ Condition2, data = sample_info, permutations = 999)
print("--- 2 Categories (Fraction Only) ---")
print(permanova_results2)
print(" ")

# --- 3. Beta Dispersion Test ---
# betadisper requires a single vector defining groups. We generate the 4 distinct 
# categories on the fly by pasting Condition1 and Condition2 together.
combined_groups <- interaction(sample_info$Condition1, sample_info$Condition2)
dispersion_test <- betadisper(dist_matrix, group = combined_groups)
anova_results <- anova(dispersion_test)
print("--- Beta Dispersion (Across 4 Conditions) ---")
print(anova_results)

## Phase 3a: Differential Expression Analysis

In [ ]:
USE_LFC_SHRINK <- TRUE  # Flip to TRUE if you wish to apply shrinkage

dds <- DESeq(dds, quiet = TRUE)

# Extract the Labeled Pulldown vs Labeled Input results
res_lab_PD_vs_Input <- results(dds, contrast = list(c("Condition2_pull_down_vs_input", 
                                                      "Condition1lab.Condition2pull_down")))
res_lab_PD_vs_Input_shrunk <- lfcShrink(dds, 
                                        contrast = list(c("Condition2_pull_down_vs_input", 
                                                          "Condition1lab.Condition2pull_down")), 
                                        type = "ashr", quiet = TRUE)

# Extract the Labeled Pulldown vs all Input results
coef_names <- resultsNames(dds)
idx_main <- which(coef_names == "Condition2_pull_down_vs_input")
idx_inter <- which(coef_names == "Condition1lab.Condition2pull_down")
if (length(idx_main) == 0 || length(idx_inter) == 0) {
    stop("Could not find the required coefficients in resultsNames(dds).")
}
all_inputs_contrast <- rep(0, length(coef_names))
all_inputs_contrast[idx_main] <- 1
all_inputs_contrast[idx_inter] <- 0.5

res_lab_PD_vs_All_Input <- results(dds, contrast = all_inputs_contrast)
res_lab_PD_vs_All_Input_shrunk <- lfcShrink(dds, 
                                            contrast = all_inputs_contrast, 
                                            type = "ashr", quiet = TRUE)

# Extract the Labeled Pulldown vs Unlabeled Pulldown results
res_lab_vs_unlab_PD <- results(dds, contrast = list(c("Condition1_lab_vs_unlab", 
                                                      "Condition1lab.Condition2pull_down")))
res_lab_vs_unlab_PD_shrunk <- lfcShrink(dds, 
                                        contrast = list(c("Condition1_lab_vs_unlab", 
                                                          "Condition1lab.Condition2pull_down")), 
                                        type = "ashr", quiet = TRUE)

categorise_expression_tiers <- function(unshrunk_res, shrunk_res, fc_cut = 0.5, padj_cut = 0.05, pval_cut = 0.05) {
    df <- data.frame(
        log2FoldChange = shrunk_res$log2FoldChange,
        padj           = unshrunk_res$padj,
        pvalue         = unshrunk_res$pvalue,
        row.names      = rownames(shrunk_res)
    )
    df <- df[!is.na(df$padj) & !is.na(df$pvalue), ]
    df$Confidence_Tier <- "Not Significant"
    
    # Tier 1: Strong LFC (> fc_cut) AND Strong Stats (padj)
    # Tier 2: Moderate LFC (0 to fc_cut) AND Strong Stats (padj)
    # Tier 3: Strong LFC (> fc_cut) AND Weaker Stats (pvalue but not padj)
    # Tier 4: Moderate LFC (0 to fc_cut) AND Weaker Stats (pvalue but not padj)

    df$Confidence_Tier[df$log2FoldChange > fc_cut & df$padj < padj_cut] <- "Tier 1: Strongly Up, Strong Stats"
    df$Confidence_Tier[df$log2FoldChange > 0 & df$log2FoldChange <= fc_cut & df$padj < padj_cut] <- "Tier 2: Moderately Up, Strong Stats"
    df$Confidence_Tier[df$log2FoldChange > fc_cut & df$padj >= padj_cut & df$pvalue < pval_cut] <- "Tier 3: Strongly Up, Weak Stats"
    df$Confidence_Tier[df$log2FoldChange > 0 & df$log2FoldChange <= fc_cut & df$padj >= padj_cut & df$pvalue < pval_cut] <- "Tier 4: Moderately Up, Weak Stats"
    
    df$Confidence_Tier[df$log2FoldChange < -fc_cut & df$padj < padj_cut] <- "Tier 1: Strongly Down, Strong Stats"
    df$Confidence_Tier[df$log2FoldChange >= -fc_cut & df$log2FoldChange < 0 & df$padj < padj_cut] <- "Tier 2: Moderately Down, Strong Stats"
    df$Confidence_Tier[df$log2FoldChange < -fc_cut & df$padj >= padj_cut & df$pvalue < pval_cut] <- "Tier 3: Strongly Down, Weak Stats"
    df$Confidence_Tier[df$log2FoldChange >= -fc_cut & df$log2FoldChange < 0 & df$padj >= padj_cut & df$pvalue < pval_cut] <- "Tier 4: Moderately Down, Weak Stats"
    
    return(df)
}

print_custom_tier_summary <- function(categorised_df) {
    # Define the mapping of internal labels to your preferred display names
    tier_mapping <- list(
        "not significant" = "Not Significant",
        "tier 1 up"     = "Tier 1: Strongly Up, Strong Stats",
        "tier 2 up"     = "Tier 2: Moderately Up, Strong Stats",
        "tier 3 up"     = "Tier 3: Strongly Up, Weak Stats",
        "tier 4 up"     = "Tier 4: Moderately Up, Weak Stats",
        "tier 1 down"   = "Tier 1: Strongly Down, Strong Stats",
        "tier 2 down"   = "Tier 2: Moderately Down, Strong Stats",
        "tier 3 down"   = "Tier 3: Strongly Down, Weak Stats",
        "tier 4 down"   = "Tier 4: Moderately Down, Weak Stats"
    )
    
    # Loop through and print the count for each category
    for (display_name in names(tier_mapping)) {
        internal_label <- tier_mapping[[display_name]]
        # Count how many genes match this specific tier string
        count <- sum(categorised_df$Confidence_Tier == internal_label)
        cat(paste0(display_name, " ", count, "\n"))
    }
}

message("--- Categorising Comparison 1a: Labeled Pulldown vs Labeled Input ---")
categorised_PD_vs_Input <- categorise_expression_tiers(
    unshrunk_res = res_lab_PD_vs_Input, 
    shrunk_res   = res_lab_PD_vs_Input_shrunk,
    fc_cut       = FC_thresh, 
    padj_cut     = FCPadj, 
    pval_cut     = FCPval
)
print_custom_tier_summary(categorised_PD_vs_Input)

message("\n--- Categorising Comparison 1b: Labeled Pulldown vs All Input ---")
categorised_PD_vs_All_Input <- categorise_expression_tiers(
    unshrunk_res = res_lab_PD_vs_All_Input, 
    shrunk_res   = res_lab_PD_vs_All_Input_shrunk,
    fc_cut       = FC_thresh, 
    padj_cut     = FCPadj, 
    pval_cut     = FCPval
)
print_custom_tier_summary(categorised_PD_vs_All_Input)

message("\n--- Categorising Comparison 2: Labeled Pulldown vs Unlabeled Pulldown ---")
categorised_lab_vs_unlab_PD <- categorise_expression_tiers(
    unshrunk_res = res_lab_vs_unlab_PD, 
    shrunk_res   = res_lab_vs_unlab_PD_shrunk,
    fc_cut       = FC_thresh, 
    padj_cut     = FCPadj, 
    pval_cut     = FCPval
)
print_custom_tier_summary(categorised_lab_vs_unlab_PD)

# --- Export the Dataframes to CSV ---
write.csv(categorised_PD_vs_Input, file = "o_outputs/processed_data/DE_results/Tiered_Results_Lab_PD_vs_Input.csv")
write.csv(categorised_lab_vs_unlab_PD, file = "o_outputs/processed_data/DE_results/Tiered_Results_Lab_vs_Unlab_PD.csv")

## Phase 3b: Volcano Plot

In [ ]:
res_df <- as.data.frame(res)
res_df <- res_df[!is.na(res_df$padj), ]

res_df$Significance <- "Not Significant"
res_df$Significance[res_df$log2FoldChange > 0.5 & res_df$padj < 0.05] <- "Up-regulated"
res_df$Significance[res_df$log2FoldChange < -0.5 & res_df$padj < 0.05] <- "Down-regulated"

volcano_plot <- ggplot(res_df, aes(x = log2FoldChange, y = -log10(padj), color = Significance)) +
    geom_point(alpha = 0.6, size = 1.5) +
    scale_color_manual(values = c("Up-regulated" = "red", "Down-regulated" = "blue", "Not Significant" = "grey")) +
    geom_vline(xintercept = c(-0.5, 0.5), linetype = "dashed", color = "black") +
    geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "black") +
    theme_minimal() +
    labs(x = "Log2 Fold Change", y = "-Log10(Adjusted P-value)") +
    theme(legend.position = "right")

print(volcano_plot)
ggsave(filename = paste0("f_figures/DE/Volcano_Plot.png"),
       plot = volcano_plot, width = 8, height = 4, dpi = 300)

## Phase 3c: MA-Plot

In [ ]:
plotMA(res, main = glue("MA Plot"), ylim = c(-4, 4))

png(glue("f_figures/MA/MA_Plot.png"), width = 2400, height = 1800, res = 300)
plotMA(res, main = glue("MA Plot"), ylim = c(-4, 4))
dev.off()

## Phase 3d: Expressed Gene Overlap (Optional/Uncommented)

In [ ]:
# files <- list(
#     "LP_vs_UP"  = "o_outputs/processed_data/DE_results/DESeq2_upreg-Results_lab_pull_down_vs_unlab_pull_down.csv",
#     "LP_vs_LI"  = "o_outputs/processed_data/DE_results/DESeq2_upreg-Results_lab_pull_down_vs_lab_input.csv"
# )
# gene_list <- lapply(files, function(f) { if (file.exists(f)) return(read.csv(f)$X) else return(character(0)) })
# all_unique_genes <- unique(unlist(gene_list))
# presence_matrix <- sapply(gene_list, function(g) all_unique_genes %in% g)
# overlap_df <- as.data.frame(presence_matrix)
# row.names(overlap_df) <- all_unique_genes
# pheatmap(as.matrix(overlap_df) + 0, cluster_rows = TRUE, cluster_cols = FALSE, show_rownames = FALSE)

## Phase 4: Network Analysis (Gene Correlation)

In [ ]:
norm_counts <- assay(vsd)
dat_expr <- t(norm_counts)
gene_correlations <- cor(dat_expr, method = "pearson")
top_var_genes <- head(order(rowVars(norm_counts), decreasing = TRUE), network_size)

network_plot <- pheatmap(norm_counts[top_var_genes, ],
                         annotation_col = sample_info,
                         show_rownames = FALSE,
                         main = paste0("Co-expression of Top ", network_size, " Variable Genes"),
                         filename = glue("f_figures/Network-coexpr/Top_", network_size, "_Coexpression_Heatmap.png"),
                         width = 8, height = 6,
                         silent = FALSE)

## Phase 5a: Gene Ontology (GO) Enrichment Analysis

In [ ]:
org_db <- org.Mm.eg.db
id_type <- "SYMBOL"

# --- 1. Select Data Source and Define Universe ---
# Choose which comparison you want to run GO enrichment on:
# Option A: categorised_PD_vs_Input
# Option B: categorised_PD_vs_All_Input
# Option C: categorised_lab_vs_unlab_PD
target_de_data <- categorised_PD_vs_All_Input

# The universe should be all genes that passed initial filtering in this specific comparison
universe_genes <- rownames(target_de_data)

# --- 2. Extract Genes by Target Confidence Tiers ---
# High Confidence Targets (Tiers 1 and 2: strong or moderate LFC with padj < 0.05)
up_genes_high_conf <- rownames(target_de_data[
    target_de_data$Confidence_Tier %in% c("Tier 1: Strongly Up, Strong Stats", 
                                          "Tier 2: Moderately Up, Strong Stats"), 
])
down_genes_high_conf <- rownames(target_de_data[
    target_de_data$Confidence_Tier %in% c("Tier 1: Strongly Down, Strong Stats", 
                                          "Tier 2: Moderately Down, Strong Stats"), 
])

# (Optional) Exploratory Targets (Tiers 3 and 4: pvalue < 0.05 but padj >= 0.05)
up_genes_weak_conf <- rownames(target_de_data[
    target_de_data$Confidence_Tier %in% c("Tier 3: Strongly Up, Weak Stats", 
                                          "Tier 4: Moderately Up, Weak Stats"), 
])
down_genes_weak_conf <- rownames(target_de_data[
    target_de_data$Confidence_Tier %in% c("Tier 3: Strongly Down, Weak Stats", 
                                          "Tier 4: Moderately Down, Weak Stats"), 
]) 

# --- 3. Run GO Enrichment on High Confidence Genes ---

message(glue("Running GO Enrichment on {length(up_genes_high_conf)} High-Confidence Up-regulated genes..."))
go_up <- enrichGO(gene          = up_genes_high_conf,
                  universe      = universe_genes,
                  OrgDb         = org_db,
                  keyType       = id_type,
                  ont           = "ALL",
                  pAdjustMethod = "BH",
                  pvalueCutoff  = 0.05,
                  qvalueCutoff  = 0.2,
                  readable      = TRUE)

message(glue("Running GO Enrichment on {length(down_genes_high_conf)} High-Confidence Down-regulated genes..."))
go_down <- enrichGO(gene        = down_genes_high_conf,
                    universe    = universe_genes,
                    OrgDb       = org_db,
                    keyType     = id_type,
                    ont         = "ALL",
                    pAdjustMethod = "BH",
                    pvalueCutoff  = 0.05,
                    qvalueCutoff  = 0.2,
                    readable      = TRUE)
up_results <- if(!is.null(go_up)) as.data.frame(go_up) else data.frame()
down_results <- if(!is.null(go_down)) as.data.frame(go_down) else data.frame()

# --- 5. Cilia Targeted Search ---
if (nrow(up_results) > 0) {
    cilia_terms <- up_results[grep("cili", up_results$Description, ignore.case = TRUE), ]
    message("Number of cilium-related terms found in High-Confidence Up targets: ", nrow(cilia_terms))
    
    if (nrow(cilia_terms) > 0) {
        cilia_genes_list <- cilia_terms$geneID |>
            paste(collapse = "/") |>
            strsplit(split = "/") |>
            unlist() |>
            unique() |>
            sort()
        
        message("Found ", length(cilia_genes_list), " unique genes matching cilia terms:")
        print(cilia_genes_list)
    }
} else {
    message("No enriched GO terms found for Up-regulated genes at current thresholds.")
}

# --- 5. Export Results ---
write.csv(up_results, file = glue("o_outputs/processed_data/GO_results/GO_Enrichment_HighConf_UP.csv"))
write.csv(down_results, file = glue("o_outputs/processed_data/GO_results/GO_Enrichment_HighConf_DOWN.csv"))

# --- 3. Run GO Enrichment on weak-Confidence Genes ---

message(glue("Running GO Enrichment on {length(up_genes_weak_conf)} Weak-Confidence Up-regulated genes..."))
go_up <- enrichGO(gene          = up_genes_weak_conf,
                  universe      = universe_genes,
                  OrgDb         = org_db,
                  keyType       = id_type,
                  ont           = "ALL",
                  pAdjustMethod = "BH",
                  pvalueCutoff  = 0.05,
                  qvalueCutoff  = 0.2,
                  readable      = TRUE)

message(glue("Running GO Enrichment on {length(down_genes_weak_conf)} Weak-Confidence Down-regulated genes..."))
go_down <- enrichGO(gene        = down_genes_weak_conf,
                    universe    = universe_genes,
                    OrgDb       = org_db,
                    keyType     = id_type,
                    ont         = "ALL",
                    pAdjustMethod = "BH",
                    pvalueCutoff  = 0.05,
                    qvalueCutoff  = 0.2,
                    readable      = TRUE)
up_results <- if(!is.null(go_up)) as.data.frame(go_up) else data.frame()
down_results <- if(!is.null(go_down)) as.data.frame(go_down) else data.frame()

# --- 5. Cilia Targeted Search ---
if (nrow(up_results) > 0) {
    cilia_terms <- up_results[grep("cili", up_results$Description, ignore.case = TRUE), ]
    message("Number of cilium-related terms found in Weak-Confidence Up targets: ", nrow(cilia_terms))
    
    if (nrow(cilia_terms) > 0) {
        cilia_genes_list <- cilia_terms$geneID |>
            paste(collapse = "/") |>
            strsplit(split = "/") |>
            unlist() |>
            unique() |>
            sort()
        
        message("Found ", length(cilia_genes_list), " unique genes matching cilia terms:")
        print(cilia_genes_list)
    }
} else {
    message("No enriched GO terms found for Up-regulated genes at current thresholds.")
}

# --- 5. Export Results ---
write.csv(up_results, file = glue("o_outputs/processed_data/GO_results/GO_Enrichment_WeakConf_UP.csv"))
write.csv(down_results, file = glue("o_outputs/processed_data/GO_results/GO_Enrichment_WeakConf_DOWN.csv"))

## Phase 5b: Gene Ontology (GO) Enrichment Plot

In [ ]:
# Define a label for your current analysis to use in titles and filenames
# Examples: "Lab_PD_vs_All_Input" or "Lab_vs_Unlab_PD"
comparison_name <- "Lab_PD_vs_All_Input"

if (!is.null(go_up) && nrow(go_up) > 0) {
    dot_up <- dotplot(go_up, showCategory = 20) +
        ggtitle(glue("Top Biological Processes: Up-regulated in {comparison_name}")) +
        theme(
            axis.text.y = element_text(size = 10),
            axis.title.y = element_blank()
        )
    print(dot_up)
    ggsave(filename = glue("f_figures/GO/GO_Dotplot_UP_{comparison_name}.png"),
           plot = dot_up, width = 10, height = 8, dpi = 300)
} else {
    message(glue("Skipping Up-regulated dotplot: No enriched terms for {comparison_name}."))
}

if (!is.null(go_down) && nrow(go_down) > 0) {
    dot_down <- dotplot(go_down, showCategory = 20) +
        ggtitle(glue("Top Biological Processes: Down-regulated in {comparison_name}")) +
        theme(
            axis.text.y = element_text(size = 10),
            axis.title.y = element_blank()
        )
    print(dot_down)
    ggsave(filename = glue("f_figures/GO/GO_Dotplot_DOWN_{comparison_name}.png"),
           plot = dot_down, width = 9, height = 7, dpi = 300)
} else {
    message(glue("Skipping Down-regulated dotplot: No enriched terms for {comparison_name}."))
}

## Phase 6a: Gene Ontology (GO) GSEA implementation

In [ ]:
# --- 1. Prepare the Ranked Gene List ---
# Convert the DESeq2 lfcShrink result into a data frame
res_df <- as.data.frame(res)

# Remove genes that have missing p-values or log2FoldChanges
res_df <- res_df[!is.na(res_df$pvalue) & !is.na(res_df$log2FoldChange), ]

# Calculate a ranking statistic: sign of LFC multiplied by -log10 of the p-value.
# High positive values = strongly up-regulated and highly significant.
# High negative values = strongly down-regulated and highly significant.
res_df$ranking_stat <- sign(res_df$log2FoldChange) * (-log10(res_df$pvalue))

# Extract the named vector required by clusterProfiler
ranked_genes <- res_df$ranking_stat
names(ranked_genes) <- rownames(res_df)

# Sort the vector in strict decreasing order (critical for GSEA)
ranked_genes <- sort(ranked_genes, decreasing = TRUE)

# --- 2. Run GSEA ---
org_db <- org.Mm.eg.db
id_type <- "SYMBOL"

message("Running Gene Set Enrichment Analysis...")
gsea_go <- gseGO(geneList      = ranked_genes,
                 OrgDb         = org_db,
                 keyType       = id_type,
                 ont           = "ALL",        # Evaluates BP, CC, and MF terms
                 pvalueCutoff  = 0.05,         # Target significance for the pathway
                 pAdjustMethod = "BH",
                 eps           = 0,            # Setting to 0 prevents p-value rounding limits
                 seed          = 123)          # Ensures reproducibility of permutation math

gsea_results <- as.data.frame(gsea_go)


# --- 3. Extract Target Pathways (e.g., Cilia) ---
cilia_terms <- gsea_results[grep("cili", gsea_results$Description, ignore.case = TRUE), ]
message("Number of cilium-related pathways identified by GSEA: ", nrow(cilia_terms))

# Write out the full table of enriched functional pathways
write.csv(gsea_results, file = glue("o_outputs/processed_data/GO_results/GSEA_GO_Enrichment.csv"))

## Phase 6b: GSEA Visualization Plots

In [ ]:
if (!is.null(gsea_go) && nrow(gsea_go) > 0) {
    
    # 1. Global Overview Dotplot
    # GSEA dotplots split pathways by whether they are activated (NES > 0) or suppressed (NES < 0)
    dot_gsea <- dotplot(gsea_go, showCategory = 15, split = ".sign") +
        facet_grid(. ~ .sign) +
        ggtitle(glue("GSEA Functional Shifts")) +
        theme(axis.text.y = element_text(size = 8))
        
    print(dot_gsea)
    ggsave(filename = glue("f_figures/GO/GSEA_Dotplot.png"),
           plot = dot_gsea, width = 11, height = 8, dpi = 300)
    
    
    # 2. Specific Target Pathway Visualisation (GSEA Plot)
    # If your target pathway (e.g., "cilium organization") shows up in your results, 
    # you can plot its exact running enrichment score mountain.
    
    # Let's find the exact ID of the top cilia-related term found
    if (nrow(cilia_terms) > 0) {
        top_cilia_id <- cilia_terms$ID[1]
        top_cilia_desc <- cilia_terms$Description[1]
        
        # Generate the classic GSEA mountain plot
        gsea_mountain <- gseaplot2(gsea_go, 
                                   geneSetID = top_cilia_id, 
                                   title = glue("Running Enrichment: {top_cilia_desc}"))
        
        print(gsea_mountain)
        ggsave(filename = glue("f_figures/GO/GSEA_Mountain_Plot_Cilia.png"),
               plot = gsea_mountain, width = 8, height = 6, dpi = 300)
    }
}